# TB-Trust — 00: Setup and smoke test

Clones the repo, installs it, and verifies the whole core runs before you touch any data. If the smoke test passes, the environment is good.

Every notebook in this series reads its paths from the configuration cell below, so nothing is hard-coded to Kaggle.

In [ ]:
# --- configuration ---------------------------------------------------------
# Defaults are the Kaggle paths. Every path is read from the environment first,
# so the same notebook runs unmodified on Kaggle, locally, or in CI -- which is
# also what lets these notebooks be executed as a test rather than only read.
import os

REPO = os.environ.get("TBTRUST_REPO", "/kaggle/working/tb-trust")
DATA = os.environ.get("TBTRUST_DATA", "/kaggle/input/tuberculosis-tb-chest-xray-dataset")
WORK = os.environ.get("TBTRUST_WORK", "/kaggle/working")
REPO_URL = os.environ.get("TBTRUST_REPO_URL", "https://github.com/AIscend-Research/tb-trust.git")

MANIFEST = f"{WORK}/manifest.csv"
OUT = f"{WORK}/outputs"
os.makedirs(WORK, exist_ok=True)
print("REPO:", REPO, "\nDATA:", DATA, "\nWORK:", WORK)

## 1. Clone and install

Set `TBTRUST_REPO_URL` (or edit `REPO_URL` above) if you are working from a fork.

In [ ]:
import subprocess
import sys
from pathlib import Path

if not Path(REPO, "pyproject.toml").exists():
    Path(REPO).parent.mkdir(parents=True, exist_ok=True)
    print(f"cloning {REPO_URL} -> {REPO}")
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO], check=True)
else:
    print("repo already present at", REPO)

# Only install if the package doesn't already resolve, so re-running this
# notebook (or running it against a checkout that is already installed) doesn't
# reinstall over a working environment.
import importlib.util

sys.path.insert(0, str(Path(REPO, "src")))
if importlib.util.find_spec("tbtrust") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", REPO], check=True)
    importlib.invalidate_caches()
    print("installed")
else:
    print("tbtrust already resolves -- skipping install")

In [ ]:
# Enter the repo and make it importable. The install is skipped when the package
# already resolves, so re-running a notebook is cheap.
import importlib.util
import os
import subprocess
import sys

os.chdir(REPO)
sys.path.insert(0, os.path.join(REPO, "src"))
if importlib.util.find_spec("tbtrust") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
    importlib.invalidate_caches()
print("tbtrust ready from", REPO)

## 2. Smoke test

Exercises the degradation pipeline, the manifest, LOCO splits, calibration, safe deferral, conformal coverage, and a torch forward/backward — on synthetic data, with no GPU and no images.

In [ ]:
import subprocess
import sys

r = subprocess.run([sys.executable, "scripts/smoke_test.py"], capture_output=True, text=True)
print(r.stdout or r.stderr)
assert r.returncode == 0, "smoke test failed"

## 3. GPU check (optional)

In [ ]:
import torch

print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    print("Running on CPU. Training will be slow; enable a GPU accelerator for the real runs.")

## 4. Test suite

In [ ]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("pytest") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev]"], check=True)
r = subprocess.run([sys.executable, "-m", "pytest", "-q"], capture_output=True, text=True)
print(r.stdout[-3000:] or r.stderr[-3000:])
assert r.returncode == 0, "pytest failed"

Next: **01_data_and_degradation_ablation.ipynb**.